In [ ]:
import pandas as pd
import yaml

import matplotlib.pyplot as plt
import mplhep as mpl
mpl.style.use('CMS')
import seaborn as sns

## Figure configs

In [ ]:
with open("./board_configs_yaml/DESY_TB_2026May_CE.yaml", "r") as file:

    fig_configs = yaml.safe_load(file)

print(fig_configs.keys())

given_run = 'run9'
selected_fig_config = fig_configs[given_run]

for id in selected_fig_config.keys():
    selected_fig_config[id]['run'] = given_run
    selected_fig_config[id]['title'] = f"{selected_fig_config[id]['name']} HV{selected_fig_config[id]['HV']}V OS:{selected_fig_config[id]['offset']}"

    try:
        selected_fig_config[id]['title'] += f' {selected_fig_config[id]['angle']}deg'
    except:
        pass

    try:
        selected_fig_config[id]['title'] += f' temp:{selected_fig_config[id]['temp']}'
    except:
        pass

    try:
        selected_fig_config[id]['title'] += f' IRRAD:{selected_fig_config[id]['irrad']}'
    except:
        pass

    try:
        selected_fig_config[id]['title'] += f' {selected_fig_config[id]['note']}'
    except:
        pass


roles = {}
for board_id, board_info in selected_fig_config.items():
    roles[board_info.get('role')] = board_id

for key, val in selected_fig_config.items():
    print(key, val)

In [ ]:
path_to_res = f'./beam_test_results/desy_May2026_results/combined_45deg_m25C_iminuit/resolution_table.csv'
resolution_df = pd.read_csv(path_to_res)

## nevt_track_df: number of events dataframe
path_to_nevt = f'./beam_test_results/desy_May2026_results/combined_45deg_m25C_iminuit/nevt.csv'
nevt_track_df = pd.read_csv(path_to_nevt)

# directory_to_save = Path('/home/jongho/tmp/plots')

In [ ]:
if nevt_track_df.columns.size == 9:
    col_to_merge = ['row_trig', 'col_trig', 'row_dut', 'col_dut', 'row_extra', 'col_extra', 'row_ref', 'col_ref']
else:
    col_to_merge = ['row_trig', 'col_trig', 'row_dut', 'col_dut', 'row_ref', 'col_ref']

merged_df = pd.merge(resolution_df, nevt_track_df, on=col_to_merge)
merged_df = merged_df.loc[merged_df['err_ref'] != 0]
merged_df.sort_values(by=['nevt'], ascending=False).reset_index(drop=True)

In [ ]:
## Remove invalid fit
for role in roles.keys():

    col_name = f'fit_valid_{role}'

    if not col_name in merged_df.columns:
        continue

    merged_df = merged_df[merged_df[col_name] == 1].reset_index(drop=True)

In [ ]:
for role in roles.keys():

    col_name = f'div_{role}'

    if not col_name in merged_df.columns:
        continue

    g = sns.jointplot(
        data=merged_df,
        x=f'div_{role}',
        y=f'red_chi2_{role}',
        kind='scatter',
        alpha=0.4,
        height=10,
        color='royalblue',
        marginal_kws=dict(bins=30, fill=True)
    )

    # Add a density contour to see where the "bulk" of the 3000 tracks lie
    g.plot_joint(sns.kdeplot, color="crimson", zorder=5, levels=5, alpha=0.5)
    g.set_axis_labels(r'Divergence $(\mu_f - \mu_{init})/\sigma$', r'Reduced $\chi^{2}$', fontsize=12)
    plt.subplots_adjust(top=0.9)
    g.figure.suptitle(f'Quality Correlation: {selected_fig_config[roles[role]].get('name')}', fontsize=15)
    plt.tight_layout()

In [ ]:
for role in roles.keys():
    col_name = f'div_{role}'
    chi2_col = f'red_chi2_{role}'

    if not col_name in merged_df.columns:
        continue

    plt.figure(figsize=(14, 6))

    # 1. Calculate the threshold values for 85%, 90%, and 95%
    # We drop NaNs to ensure quantile calculation works
    targets = [0.85, 0.90, 0.95]
    thresholds = merged_df[chi2_col].dropna().quantile(targets)

    # Plot the CDFs
    sns.ecdfplot(data=merged_df, x=chi2_col, label=r'Reduced $\chi^{2}$', color='forestgreen', lw=2)
    sns.ecdfplot(data=merged_df, x=col_name, label='Divergence', color='orange', lw=2)

    # 2. Add intersection lines for the target efficiencies
    colors = ['blue', 'red', 'darkred']
    for q, val, col in zip(targets, thresholds, colors):
        # Draw horizontal line at the efficiency level
        plt.axhline(q, color=col, linestyle='--', alpha=0.3)

        # Draw vertical line at the corresponding Chi2 value
        plt.axvline(val, color=col, linestyle='--', alpha=0.6,
                    label=rf'{int(q*100)}% Eff: $\chi^{2} < {val:.2f}$')

        # Optional: Add a text label near the intersection
        plt.text(val + 0.01, 0.75, f' {val:.2f}', color=col, fontweight='bold', fontsize=15)

    plt.xlim(-0.2, 2.5)
    plt.ylim(0, 1.05)
    plt.xlabel(r'Reduced $\chi^{2}$', fontsize=14)
    plt.ylabel('Cumulative Probability', fontsize=14)
    plt.title(f'Efficiency vs. Quality Thresholds ({selected_fig_config[roles[role]].get("name")})', fontsize=16)

    # Place legend to the side or bottom if it gets too crowded
    plt.legend(loc='center', fontsize=16, bbox_to_anchor=(1.14,0.5))
    plt.grid(True, which="both", ls="-", alpha=0.2)
    plt.tight_layout()

In [ ]:
for role in roles.keys():
    col_name = f'div_{role}'
    if not col_name in merged_df.columns:
        continue

    # 1. Define your data columns as you already have
    cols = [
        f'res_{role}',
        f'err_{role}',
        f'rel_error_{role}',
        f'red_chi2_{role}',
        f'div_{role}',
        'nevt',
    ]

    # 2. Define a list of "Clean" labels for the plot
    clean_labels = ['res', 'err', 'rel\nerror', 'red\nchi2', 'div', 'nevt']

    corr = merged_df[cols].corr()

    plt.figure(figsize=(10, 9))
    sns.heatmap(
        corr,
        annot=True,
        cmap='coolwarm',
        fmt=".2f",
        linewidths=0.5,
        vmin=-1, vmax=1,
        # 3. Apply the clean labels here
        xticklabels=clean_labels,
        yticklabels=clean_labels
    )

    plt.title(f'Correlation Matrix: {selected_fig_config[roles[role]].get("name")}', fontsize=20)
    plt.tight_layout()

In [ ]:
for role in roles.keys():

    col_name = f'res_{role}'

    if not col_name in merged_df.columns:
        continue

    plt.figure(figsize=(12, 8))

    # Scatter plot colored by divergence to see 3-way relationship
    sns.scatterplot(
        data=merged_df,
        x=f'res_{role}',       # Resolution
        y=f'red_chi2_{role}',  # Quality
        hue=f'div_{role}',     # Stability
        palette='magma',
        alpha=0.5,
        edgecolor=None
    )

    # FIX: Use line_kws to pass linestyle to the underlying plotter
    sns.regplot(
        data=merged_df,
        x=f'res_{role}',
        y=f'red_chi2_{role}',
        scatter=False,
        color='black',
        line_kws={'ls': '--', 'lw': 2}
    )

    plt.xlabel(r'Measured Resolution $\sigma$ [ps]', fontsize=14)
    plt.ylabel(r'Reduced $\chi^{2}$', fontsize=14)
    plt.title(f'Fit Bias Check ({selected_fig_config[roles[role]].get('name')})', fontsize=16)
    plt.tight_layout()